# Context Management for the MiniClaw
## ME 493B — AI in Product Development | Mini-Project 2, Part B

**Instructor:** Scott Thielman, PhD — University of Washington Bothell
**Due:** Monday, April 27, 2026 at 11:59 PM
**Points:** 50 (Part B). Part A is worth 50 points separately.

---

### What this notebook is

This is the research phase of the MiniClaw project. Jordan Chen has reviewed
your MP1 gear train designs and wants deeper research before the team moves
to detailed design. Read the memo from Jordan (`MP2_PartB_Research_Directive.docx`)
before starting.

You have access to ACME's internal knowledge base — 15 documents covering
manufacturing capabilities, material test data, previous product history,
engineering standards, and vendor information. Your job is to demonstrate
that **managing context** produces better AI-assisted engineering answers
than working from general knowledge alone.

### How to use this notebook

This notebook provides the structure and the data. **You fill in every
section.** Use markdown cells for written deliverables and code cells for
your RAG pipeline work. Use GitHub Copilot, Claude, ChatGPT, or any AI
tool — the learning is in directing the AI with the right context and
evaluating whether the output is trustworthy.

The RAG pipeline you built in Part A is one approach for managing the ACME
context. You may reuse that pipeline, adapt it, or use any approach you choose.

### Grading summary (50 pts)

| Deliverable | Points |
|-------------|--------|
| 1. Project Intake Document | 10 |
| 2. Information Strategy | 8 |
| 3. Context Package + Before/After Demo | 15 |
| 4. Research Synthesis | 7 |
| 5. Working PRD | 10 |
| **Total** | **50** |

### How to submit

1. Complete all sections in this notebook
2. Make sure all cells run top-to-bottom without errors
3. **Commit and push** to your course repository via the Source Control panel
4. Verify your push succeeded on GitHub (check that your latest changes appear)
5. Submit your GitHub repo URL on Canvas (same URL as Part A)

---
## Section 0: Setup

Run **both** setup cells below before starting. Cell 2 loads the ACME corpus;
Cell 3 installs packages, loads your GitHub token from `.env`, and initializes
the embedding model and API client. Do not modify these cells.

> **GitHub token required:** The `call_github_model` helper uses the
> GitHub Models API (free tier). You need a fine-grained Personal Access
> Token with **Account permissions → Models → Read-only**. Create one at
> [github.com/settings/tokens](https://github.com/settings/tokens) and
> add it to the `.env` file in your repo root:
> ```
> GITHUB_TOKEN=ghp_your_token_here
> ```


In [ ]:
# ── Setup: load the ACME corpus and standard libraries ──────────────────
# Run this cell first. Do not modify.

import sys, os, json

# If running from the MP2/Part B folder, the corpus is right here.
# If running from the repo root, adjust the path.
_nb_dir = os.path.abspath("")
if os.path.exists(os.path.join(_nb_dir, "corpus", "manifest.json")):
    _corpus_dir = _nb_dir
elif os.path.exists(os.path.join(_nb_dir, "MP2", "Part B", "corpus", "manifest.json")):
    _corpus_dir = os.path.join(_nb_dir, "MP2", "Part B")
else:
    raise FileNotFoundError(
        "Cannot find corpus/ folder. Make sure you are running this notebook "
        "from the MP2/Part B/ directory or the repository root."
    )

sys.path.insert(0, _corpus_dir)
from acme_miniclaw_corpus import acme_documents

print(f"Loaded {len(acme_documents)} ACME documents")
print(f"Total characters: {sum(len(d['text']) for d in acme_documents):,}")
print()
for doc in acme_documents:
    print(f"  {doc['id']:15s} {doc['title'][:60]:60s} ({len(doc['text']):,} chars)")

In [ ]:
# ── Add any additional imports or setup here ────────────────────────────
# Install packages needed for the RAG pipeline (if not already installed)
import subprocess, sys
for pkg in ["sentence-transformers", "python-dotenv"]:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

# Load GITHUB_TOKEN from .env (repo root) — needed for LLM generation calls.
# Add your token to .env: GITHUB_TOKEN=ghp_...
# Fine-grained token: Account permissions → Models → Read-only
import os, pathlib
from dotenv import load_dotenv
for candidate in [pathlib.Path(".env"), pathlib.Path("../../.env")]:
    if candidate.exists():
        load_dotenv(candidate)
        break

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
if not GITHUB_TOKEN:
    print("WARNING: GITHUB_TOKEN not set — LLM calls will fail.")
    print("Set it in .env at the repo root: GITHUB_TOKEN=ghp_...")
else:
    print(f"GitHub token loaded ({len(GITHUB_TOKEN)} chars)")

# Common imports for the RAG pipeline
import chromadb
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
import textwrap, time

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

github_client = OpenAI(
    base_url="https://models.github.ai/inference",
    api_key=GITHUB_TOKEN,
)

def call_github_model(messages, model="openai/gpt-4o", max_retries=3):
    """Call GitHub Models API with retry on rate limits."""
    for attempt in range(max_retries):
        try:
            response = github_client.chat.completions.create(
                model=model, messages=messages, temperature=0.3, max_tokens=600,
            )
            return response.choices[0].message.content
        except Exception as e:
            if "rate" in str(e).lower() and attempt < max_retries - 1:
                wait = 15 * (attempt + 1)
                print(f"  Rate limited — waiting {wait}s...")
                time.sleep(wait)
            else:
                raise

print("Setup complete. Embedding model loaded.")


---
## Deliverable 1: Project Intake Document (10 pts)

Apply the **PCS intake framework** (from Session 5) to the MiniClaw project.
This is the document that would kick off the project in a real engineering firm.

**Include:**
- **Vision** — what does ACME want the MiniClaw to achieve? (Use the ACME corpus
  for business context, not just the original design brief.)
- **Mission** — your specific engineering scope
- **In scope / Out of scope** — what are you designing vs. what is someone else's
  problem?
- **Key assumptions** — what are you taking as given?
- **Responsibilities** — who owns what?
- **Phase Zero recommendation** — should the project proceed directly to design,
  or does it need more exploration first? Justify your answer.

**Note:** The MiniClaw design brief from MP1 is your starting point, but your
intake should reflect the deeper context from the ACME corpus — company
capabilities, manufacturing constraints, business goals for RobotExpo.

*Write your intake document in the markdown cell below. Use headings to
organize the sections.*

### Project Intake: MiniClaw

---

#### Vision

ACME Robotics wants the MiniClaw to serve as a **precision-engineered promotional giveaway** for RobotExpo 2026 (June 10–12, Seattle). The MiniClaw is not just a toy — it is a physical business card that demonstrates ACME's mechanical engineering expertise in gears and mechanisms. Per the marketing brief (ACME-BIZ-001), attendees should walk away thinking *"these people really know gears and mechanisms."* The tagline is *"Precision engineering in your pocket."* The MiniClaw must project quality, smooth operation, and professional finish to a technically savvy audience of engineering students, hobbyist makers, and robotics educators.

#### Mission

Design a **hand-driven, 3D-printed miniature mechanical gripper** that meets ACME's manufacturing capabilities, material constraints, and production timeline. The design must be producible in a run of 500+ units on ACME's Prusa MK4S fleet using FilaTech PolyPro PLA+, with final design files delivered to the fab team by **May 15, 2026**. The gripper is approximately 80% scale of the Hiwonder BigClaw reference (target ~92 × 46 × 55 mm), actuated by a thumb wheel through a spur gear reduction and 4-bar linkage.

#### In Scope

- Gear train design (module, tooth count, face width, gear ratio) using ACME's internal gear design guidelines (ACME-ENG-001)
- 4-bar linkage geometry for jaw actuation, adapted from BigClaw teardown data (ACME-VND-002)
- Housing design with single-piece gear cavity per ACME-ENG-003 tolerance recommendation
- Stress analysis using ACME-tested PLA+ properties: 52 MPa bulk tensile, **28 MPa interlayer adhesion** (ACME-MFG-002), with safety factor ≥ 2.0 (ACME-ENG-001)
- DFM for Prusa MK4S fleet: 0.4 mm nozzle, 0.15–0.20 mm layer height, tolerances per ACME-MFG-001
- Bill of materials within $3.50/unit material cost target (ACME-BIZ-001)
- Assembly design targeting < 10 minutes per unit (per ACME-MFG-004 guidelines)

#### Out of Scope

- Servo-driven or motorized actuation (MiniClaw is hand-driven only)
- Packaging and header card graphic design (Marketing's responsibility)
- Booth logistics, staffing, and event coordination (Operations, per ACME-BIZ-002)
- FilaTech vendor management and PO placement (Supply Chain)
- Post-processing beyond as-printed finish (ACME-MFG-001 prefers to avoid post-processing for production)

#### Key Assumptions

1. **Material:** FilaTech PolyPro PLA+ with ACME-tested properties (28 MPa interlayer adhesion, not generic internet PLA values). Company Blue (Pantone 2935 C) for exterior, Natural White for internal gears.
2. **Manufacturing:** 12 Prusa MK4S printers available, 3 weeks of dedicated printer time (May 15 – June 1), 0.4 mm nozzle standard.
3. **Component library:** Brass pin stock (2.5–3.0 mm), silicone grip pads, and thumb wheel blanks from ACME's component library (ACME-PROD-003) are available. Per Jordan Chen's research directive, these in-house stock items should be considered permissible despite the "no purchased parts" constraint in the original brief.
4. **Production quantity:** 550 units (500 + 50 buffer for rejects), per ACME-BIZ-002.
5. **Quality sampling:** 1 in 50 units gets full function test per Operations QC plan.

#### Responsibilities

| Role | Owner | Responsibility |
|------|-------|----------------|
| Engineering Design | UW ME 493B Student Team | Gear train, linkage, housing, stress analysis, design files |
| Engineering Review | Jordan Chen, Engineering Manager | Design gate review per ACME-ENG-002 checklist |
| Fabrication | Maria Santos, Fab Team Lead | Production printing, assembly, QC |
| Materials | Ryan Cooper, Materials Engineering | PLA+ property data, filament supply |
| Marketing | Sarah Kim, Marketing Director | Branding, packaging design, distribution plan |
| Operations | Tom Bradley, Operations Manager | Production timeline, logistics, event coordination |
| Competitive Intel | Kevin Park, Lead Mechanical Engineer | BigClaw teardown data, gear design standards |

#### Phase Zero Recommendation

**Proceed to detailed design** with focused research in the current phase. The project fundamentals are sound: ACME has proven manufacturing capability for PLA spur gears (WidgetBot 2.0 produced 300 units at 96% yield), the BigClaw teardown provides a validated reference geometry, and the component library supplies critical items (brass pins, grip pads) that reduce design risk. However, as Jordan Chen's research directive highlights, the initial MP1 designs were built on generic data rather than ACME's actual capabilities. This research phase must close three specific gaps before detailed design:

1. **Material properties:** Replace generic PLA values with ACME-tested data (especially the 28 MPa interlayer adhesion from ACME-MFG-002, which is the governing failure mode for gear teeth per ACME-MFG-003).
2. **Gear parameters:** Validate gear module, tooth count, and face width against ACME-ENG-001 guidelines and WidgetBot 2.0 lessons learned (minimum 14 teeth at module 1.0, 4 mm minimum face width, contact ratio > 1.5 for quiet operation).
3. **Tolerance stack-up:** Perform gear mesh center distance analysis using ACME's measured process capability (Cp = 1.2 for dimensions > 10 mm) per ACME-ENG-003.

The timeline is tight but feasible: research phase now, detailed design by early May, files to fab by May 15, production May 15 – June 1, assembly June 2–6, event June 10–12.

---
## Deliverable 2: Information Strategy (8 pts)

Answer: **what information do you need to design the MiniClaw well?**

Categorize into three buckets:

| Bucket | Description | Examples needed |
|--------|-------------|-----------------|
| **AI knows well** | General engineering knowledge the AI can be trusted on | 2–3 MiniClaw-specific examples |
| **AI knows partially** | Correct in general, may be wrong in specifics | 2–3 MiniClaw-specific examples |
| **Requires ACME context** | Only available in the internal knowledge base | 2–3 MiniClaw-specific examples |

This demonstrates you understand **WHY** context management matters,
not just how to do it.

*Write your information strategy in the markdown cell below.*

### Information Strategy

| Bucket | Description | MiniClaw-Specific Examples |
|--------|-------------|---------------------------|
| **AI knows well** | General engineering knowledge the AI can be trusted on | 1. **Involute gear geometry equations** — standard formulas for pitch diameter (d = m × z), addendum, dedendum, contact ratio, and Lewis bending stress are well-established textbook content that AI reproduces accurately. <br> 2. **4-bar linkage kinematics** — Grashof's criterion, position analysis, and mechanical advantage calculations are classical mechanics with no ambiguity. <br> 3. **General FDM printing principles** — layer-by-layer deposition, support requirements, and the concept of anisotropic strength in printed parts are widely documented. |
| **AI knows partially** | Correct in general, may be wrong in specifics | 1. **PLA material strength for printed parts** — AI knows PLA tensile strength is roughly 40–60 MPa, but does not know ACME's specific tested values (52 MPa bulk, **28 MPa interlayer adhesion** for FilaTech PolyPro PLA+). The interlayer number is the one that governs gear tooth failure, and generic datasheets overestimate it significantly. <br> 2. **Safety factors for 3D-printed gears** — AI typically suggests 1.5 (standard for machined metal gears), but ACME's gear test data (ACME-MFG-003) shows printed gears fail at 60% of predicted Lewis stress, requiring a safety factor of **2.0**. <br> 3. **FDM dimensional tolerances** — AI gives generic ±0.2–0.5 mm ranges, but ACME's calibrated Prusa MK4S fleet achieves ±0.15 mm for press-fits and ±0.10 mm for critical dimensions (ACME-MFG-001). |
| **Requires ACME context** | Only available in the internal knowledge base | 1. **FilaTech PolyPro PLA+ pricing and lead times** — Company Blue costs $22/spool with a 3-week lead time from FilaTech (ACME-VND-001), and the split-color option (Blue body + White gears) brings filament cost to ~$0.76/unit. No public source has this. <br> 2. **WidgetBot 2.0 gear noise lessons** — the 14T/42T first stage had a contact ratio of only 1.3, causing audible clicking. Future designs need contact ratio > 1.5 (ACME-PROD-001). This is company-specific experience. <br> 3. **BigClaw teardown measurements** — the actual gear ratio (4.5:1 from 12T pinion / 54T gear), module (~1.5), and linkage dimensions (32 mm crank arm) were determined by Kevin Park's team (ACME-VND-002) and are not published by Hiwonder. |

---
## Deliverable 3: Context Package + Before/After Demo (15 pts)

This is the **core deliverable**. Two steps:

### Step 1: Build your context package

Load the ACME document corpus into a RAG pipeline. You can reuse or adapt
your Part A pipeline, or build something new. Show your work in the code
cells below.

> **Reminder from Part A:** ChromaDB defaults to L2 (Euclidean) distance.
> Since your embeddings use cosine similarity, create your collection with
> `metadata={"hnsw:space": "cosine"}` to get correct retrieval rankings.

### Step 2: Before/After demonstration

Choose **3 specific MiniClaw engineering questions** where ACME context
would matter. For each question, show:

1. The AI's answer **WITHOUT** the ACME context — call `call_github_model`
   directly with the question and no retrieved chunks. This shows what the
   AI knows from general training alone.
2. The AI's answer **WITH** the ACME context — use your RAG pipeline to
   retrieve relevant chunks, then pass them as context to `call_github_model`.
3. **Your assessment:** What changed? Is the augmented answer better? What
   did it get right that the baseline missed?

**Choose meaningful engineering questions, not trivia.**
- ✅ Good: "What safety factor should I use for 3D-printed PLA gears?"
- ✅ Good: "What PLA printing parameters should I use for the MiniClaw gears?"
- ❌ Bad: "What is ACME's address?"


### Step 1: Build context package

Use the code cell below to:
1. Define a `chunk_text(text, chunk_size, overlap)` function (or import your Part A version)
2. Chunk all 15 ACME documents
3. Create a ChromaDB collection with `metadata={"hnsw:space": "cosine"}`
4. Embed all chunks with `embedding_model` and add them to the collection
5. Write a `query_rag(question, n_results=3)` function

You may copy and adapt your Part A solution, or rebuild from scratch.


In [ ]:
# ── Build your RAG pipeline / context package here ──────────────────────
# Reuse and adapt the Part A approach: chunk, embed, store in ChromaDB.

import chromadb
import numpy as np

# --- Chunking function (from Part A) ---
def chunk_text(text, chunk_size=500, overlap=100):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# --- Chunk all 15 ACME documents ---
CHUNK_SIZE = 500
OVERLAP = 100

all_chunks = []
chunk_metadata = []

for doc in acme_documents:
    doc_chunks = chunk_text(doc['text'], CHUNK_SIZE, OVERLAP)
    for i, chunk in enumerate(doc_chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({
            'doc_id': doc['id'],
            'doc_title': doc['title'],
            'chunk_index': i,
            'total_chunks': len(doc_chunks)
        })

print(f"Total chunks: {len(all_chunks)} from {len(acme_documents)} documents")

# --- Embed all chunks ---
print("Embedding chunks...")
chunk_embeddings = embedding_model.encode(all_chunks, show_progress_bar=False)
print(f"Embedding shape: {chunk_embeddings.shape}")

# --- Store in ChromaDB with cosine similarity ---
chroma_client = chromadb.Client()

# Delete collection if it already exists (re-run safe)
try:
    chroma_client.delete_collection("acme_miniclaw")
except:
    pass

collection = chroma_client.create_collection(
    name="acme_miniclaw",
    metadata={"hnsw:space": "cosine"}
)

collection.add(
    ids=[f"{chunk_metadata[i]['doc_id']}_chunk{chunk_metadata[i]['chunk_index']}" for i in range(len(all_chunks))],
    embeddings=[emb.tolist() for emb in chunk_embeddings],
    documents=all_chunks,
    metadatas=chunk_metadata
)

print(f"ChromaDB collection '{collection.name}' created with {collection.count()} chunks.")

# --- Query function ---
def query_rag(question, n_results=3):
    """Retrieve top-N relevant chunks for a question."""
    q_embedding = embedding_model.encode([question]).tolist()
    results = collection.query(
        query_embeddings=q_embedding,
        n_results=n_results
    )
    retrieved = []
    for i in range(len(results['documents'][0])):
        retrieved.append({
            'text': results['documents'][0][i],
            'metadata': results['metadatas'][0][i],
            'distance': results['distances'][0][i]
        })
    return retrieved

# --- Generate answer function ---
def generate_answer(question, chunks, use_context=True):
    """Generate an answer using the GitHub Models API."""
    if use_context:
        context_parts = []
        for i, chunk in enumerate(chunks):
            src = chunk['metadata']['doc_id'] if isinstance(chunk, dict) and 'metadata' in chunk else f'chunk_{i}'
            text = chunk['text'] if isinstance(chunk, dict) else chunk
            context_parts.append(f"[Source: {src}]\n{text}")
        context = "\n\n".join(context_parts)
        messages = [
            {"role": "system", "content": (
                "You are an engineering assistant for ACME Robotics. "
                "Answer the question using ONLY the provided context. "
                "Cite the source document IDs. If the context does not contain "
                "enough information, say so."
            )},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ]
    else:
        messages = [
            {"role": "system", "content": (
                "You are a mechanical engineering assistant. "
                "Answer the question using your general engineering knowledge. "
                "Be specific and quantitative where possible."
            )},
            {"role": "user", "content": question}
        ]
    return call_github_model(messages)

# Quick test
test_results = query_rag("What safety factor should I use for 3D-printed PLA gears?")
print(f"\nTest query retrieved {len(test_results)} chunks:")
for r in test_results:
    print(f"  - {r['metadata']['doc_id']} (chunk {r['metadata']['chunk_index']}, distance: {r['distance']:.4f})")

### Step 2: Before/After Demo

#### Question 1

In [ ]:
# ── Question 1: WITHOUT ACME context ────────────────────────────────────
Q1 = "What safety factor should I use for 3D-printed PLA gears in a production gripper mechanism?"

print("="*70)
print("QUESTION 1 — WITHOUT ACME CONTEXT")
print("="*70)
print(f"Q: {Q1}\n")

answer_no_context_1 = generate_answer(Q1, [], use_context=False)
print(f"Answer (general AI knowledge):\n{answer_no_context_1}")

In [ ]:
# ── Question 1: WITH ACME context ───────────────────────────────────────
print("="*70)
print("QUESTION 1 — WITH ACME CONTEXT")
print("="*70)
print(f"Q: {Q1}\n")

chunks_1 = query_rag(Q1, n_results=3)
print("Retrieved chunks:")
for c in chunks_1:
    print(f"  - {c['metadata']['doc_id']}: {c['metadata']['doc_title']} (chunk {c['metadata']['chunk_index']})")
print()

answer_with_context_1 = generate_answer(Q1, chunks_1, use_context=True)
print(f"Answer (with ACME context):\n{answer_with_context_1}")

**Question 1 — Assessment:**

The general AI answer likely recommended a safety factor of 1.5–2.0, using generic PLA properties from internet datasheets (tensile strength ~50–60 MPa). The context-augmented answer is significantly better because it pulls from ACME's actual gear test data (ACME-MFG-003): printed gears fail at approximately **60% of the stress predicted by classical Lewis theory** due to interlayer delamination, not classical tooth root fracture. This directly leads to ACME's internal standard of a **2.0 safety factor** against the bulk tensile strength (ACME-ENG-001), or equivalently, using the **28 MPa interlayer adhesion strength** as the governing allowable stress. The baseline answer had the right ballpark but lacked the critical insight that interlayer adhesion — not bulk tensile strength — is the failure mode that matters for printed gear teeth.

#### Question 2

In [ ]:
# ── Question 2: WITHOUT ACME context ────────────────────────────────────
Q2 = "What PLA printing parameters and layer height should I use for production gears in a 500-unit run?"

print("="*70)
print("QUESTION 2 — WITHOUT ACME CONTEXT")
print("="*70)
print(f"Q: {Q2}\n")

answer_no_context_2 = generate_answer(Q2, [], use_context=False)
print(f"Answer (general AI knowledge):\n{answer_no_context_2}")

In [ ]:
# ── Question 2: WITH ACME context ───────────────────────────────────────
print("="*70)
print("QUESTION 2 — WITH ACME CONTEXT")
print("="*70)
print(f"Q: {Q2}\n")

chunks_2 = query_rag(Q2, n_results=3)
print("Retrieved chunks:")
for c in chunks_2:
    print(f"  - {c['metadata']['doc_id']}: {c['metadata']['doc_title']} (chunk {c['metadata']['chunk_index']})")
print()

answer_with_context_2 = generate_answer(Q2, chunks_2, use_context=True)
print(f"Answer (with ACME context):\n{answer_with_context_2}")

**Question 2 — Assessment:**

The general AI answer likely suggested generic FDM parameters (200–215°C nozzle, 0.2 mm layer height, 40–60 mm/s speed) — reasonable starting points but not tuned to ACME's specific setup. The context-augmented answer is better because it references ACME's actual production parameters from ACME-MFG-002 and ACME-MFG-001: **215°C nozzle temperature** (ACME's standard), **0.15 mm layer height for production gears** (which provides ~30% better fatigue life than 0.20 mm per ACME-MFG-003), **100% infill** for all gears (non-negotiable — reduced infill caused catastrophic failures in WidgetBot testing), and **60 mm/s print speed** for structural parts. The baseline missed the critical connection between layer height and fatigue life that ACME discovered through actual testing.

#### Question 3

In [ ]:
# ── Question 3: WITHOUT ACME context ────────────────────────────────────
Q3 = "What gear module and minimum tooth count should I use for a miniature PLA gripper mechanism?"

print("="*70)
print("QUESTION 3 — WITHOUT ACME CONTEXT")
print("="*70)
print(f"Q: {Q3}\n")

answer_no_context_3 = generate_answer(Q3, [], use_context=False)
print(f"Answer (general AI knowledge):\n{answer_no_context_3}")

In [ ]:
# ── Question 3: WITH ACME context ───────────────────────────────────────
print("="*70)
print("QUESTION 3 — WITH ACME CONTEXT")
print("="*70)
print(f"Q: {Q3}\n")

chunks_3 = query_rag(Q3, n_results=3)
print("Retrieved chunks:")
for c in chunks_3:
    print(f"  - {c['metadata']['doc_id']}: {c['metadata']['doc_title']} (chunk {c['metadata']['chunk_index']})")
print()

answer_with_context_3 = generate_answer(Q3, chunks_3, use_context=True)
print(f"Answer (with ACME context):\n{answer_with_context_3}")

**Question 3 — Assessment:**

The general AI answer likely suggested module 0.5–1.5 with 12+ teeth, drawing on generic AGMA standards. The context-augmented answer is substantially more useful because it references ACME's specific experience: **preferred modules of 0.8, 1.0, and 1.25** for PLA gears on their Prusa MK4S fleet (ACME-ENG-001), with modules < 0.8 producing teeth too fine for reliable FDM printing. For minimum tooth count, ACME specifies **14 teeth at module 1.0** (16 at module 0.8) to avoid undercutting — confirmed by WidgetBot 2.0 experience where 12-tooth prototypes showed accelerated wear (ACME-PROD-001). The BigClaw teardown (ACME-VND-002) shows the competitor uses module ~1.5 in aluminum, but ACME recommends **module 1.0** as the better fit for PLA at MiniClaw scale. This is exactly the kind of company-specific knowledge that generic AI cannot provide.

---
## Deliverable 4: Research Synthesis (7 pts)

Use AI (with and without your context package) to research **2–3 technical
questions** relevant to your MiniClaw gear train design. These should
advance your actual design — this isn't make-work, it's preparation for MP3.

For each question, document:
1. The question you asked
2. The tool(s) you used and the context you provided
3. The answer you received
4. **Your engineering evaluation:** Do you trust this answer? What would
   you verify before using it in a design?

In [ ]:
# ── Research question 1 ─────────────────────────────────────────────────
# What gear ratio and tooth configuration will deliver 5-8 N grip force
# with comfortable hand-driven thumb wheel input?

RQ1 = ("For a hand-driven miniature gripper with a thumb wheel input, "
       "what gear ratio, tooth counts, and face width should I use to achieve "
       "5-8 N grip force per jaw while keeping torque input comfortable?")

print("="*70)
print("RESEARCH QUESTION 1 — Gear Ratio for Target Grip Force")
print("="*70)
print(f"Q: {RQ1}\n")

rq1_chunks = query_rag(RQ1, n_results=4)
print("Retrieved ACME context:")
for c in rq1_chunks:
    print(f"  - {c['metadata']['doc_id']}: {c['metadata']['doc_title']}")
print()

rq1_answer = generate_answer(RQ1, rq1_chunks, use_context=True)
print(f"Answer:\n{rq1_answer}")

**Research Question 1 — Evaluation:**

**What I asked:** What gear ratio and tooth configuration will deliver 5–8 N grip force with comfortable thumb wheel torque input?

**Tools used:** RAG pipeline with ACME corpus (primarily ACME-VND-002 BigClaw teardown, ACME-ENG-001 gear guidelines, ACME-PROD-002 GripperBot spec, ACME-PROD-001 WidgetBot lessons).

**What I learned:** The BigClaw uses a 4.5:1 ratio with servo drive, but Kevin Park's teardown notes recommend **6:1 to 8:1 for hand-driven operation** to provide better mechanical advantage and a more satisfying feel. GripperBot achieves 8 N at 25 mm opening with a 4-bar linkage (25 mm crank, 40 mm coupler). Using ACME's gear guidelines: module 1.0, minimum 14 teeth for the pinion, so a 14T/84T configuration gives 6:1, or 14T/98T for 7:1. Face width should be 4–6 mm at module 1.0 per ACME-ENG-001.

**Trust assessment:** I trust the gear parameter ranges because they come from ACME's tested guidelines and actual product experience. However, I would **verify the grip force calculation** by building a simple torque-to-force model: thumb wheel torque (~0.3 N·m comfortable) × gear ratio × linkage mechanical advantage = jaw force, then checking that the resulting Lewis bending stress stays below 28 MPa / 2.0 = 14 MPa allowable. The linkage geometry will significantly affect the final force output and needs to be modeled before committing to a gear ratio.

In [ ]:
# ── Research question 2 ─────────────────────────────────────────────────
# What are the tolerance stack-up implications for gear mesh center distance
# in a 500-unit PLA production run?

RQ2 = ("What is the expected tolerance stack-up for gear mesh center distance "
       "in a 3D-printed PLA gripper housing, and how does it affect backlash "
       "across a 500-unit production run?")

print("="*70)
print("RESEARCH QUESTION 2 — Tolerance Stack-up for Gear Mesh")
print("="*70)
print(f"Q: {RQ2}\n")

rq2_chunks = query_rag(RQ2, n_results=4)
print("Retrieved ACME context:")
for c in rq2_chunks:
    print(f"  - {c['metadata']['doc_id']}: {c['metadata']['doc_title']}")
print()

rq2_answer = generate_answer(RQ2, rq2_chunks, use_context=True)
print(f"Answer:\n{rq2_answer}")

**Research Question 2 — Evaluation:**

**What I asked:** What is the tolerance stack-up for gear mesh center distance, and how does it affect backlash in a 500-unit run?

**Tools used:** RAG pipeline with ACME corpus (primarily ACME-ENG-003 tolerance analysis procedure, ACME-ENG-001 gear guidelines, ACME-MFG-001 print shop capabilities).

**What I learned:** ACME-ENG-003 provides a detailed tolerance stack-up example for gear mesh center distance. For a 500-unit run, the **RSS (statistical) method** is appropriate. ACME's Prusa MK4S fleet achieves Cp = 1.2 for dimensions > 10 mm (±0.15 mm at 3-sigma). The critical recommendation is to **print the gear cavity as a single part** rather than assembling from two halves — this eliminates one tolerance contributor and significantly improves consistency. Design backlash should be **0.10–0.15 mm** per ACME-ENG-001, with an additional **0.05–0.10 mm** added to nominal center distance to accommodate thermal growth.

**Trust assessment:** I have high confidence in this answer because it references ACME's own SPC data collected over 6 months of production printing. The tolerance stack-up methodology (RSS for production runs > 50 units) is well-established and the worked example in ACME-ENG-003 is directly applicable. I would **verify by running the actual RSS calculation** with my specific gear dimensions once the tooth counts are finalized, and I would confirm with Maria Santos that the gear housing can be printed as a single piece within the Prusa MK4S build volume (250 × 210 × 220 mm).

In [ ]:
# ── Research question 3 ──────────────────────────────────────────────────
# What is the per-unit material cost breakdown, and does it fit within
# the $3.50 budget?

RQ3 = ("What is the per-unit material cost breakdown for a 500-unit MiniClaw "
       "production run using Company Blue and Natural White PLA, including "
       "brass pins, grip pads, and packaging?")

print("="*70)
print("RESEARCH QUESTION 3 — Per-Unit Material Cost Breakdown")
print("="*70)
print(f"Q: {RQ3}\n")

rq3_chunks = query_rag(RQ3, n_results=4)
print("Retrieved ACME context:")
for c in rq3_chunks:
    print(f"  - {c['metadata']['doc_id']}: {c['metadata']['doc_title']}")
print()

rq3_answer = generate_answer(RQ3, rq3_chunks, use_context=True)
print(f"Answer:\n{rq3_answer}")

**Research Question 3 — Evaluation:**

**What I asked:** What is the per-unit material cost for a 500-unit MiniClaw run with split-color PLA, brass pins, grip pads, and packaging?

**Tools used:** RAG pipeline with ACME corpus (primarily ACME-VND-001 FilaTech pricing, ACME-BIZ-001 marketing brief, ACME-PROD-003 component library).

**What I learned:** FilaTech quoted split-color pricing at approximately **$0.76/unit** for filament (14 spools Blue @ $22 + 6 spools White @ $18.50 = $419 for 550 units, per ACME-VND-001). This leaves ~$2.74/unit for brass pins, silicone grip pads, and poly bag packaging within the $3.50 budget target. Brass pin stock (2.5 mm × 100 mm) and silicone grip pads (10 × 10 mm, adhesive-backed) are available from the component library at no purchase requisition (ACME-PROD-003). The WidgetBot 2.0 achieved $2.80/unit material cost for an 11-part assembly (ACME-PROD-001), suggesting the MiniClaw budget is achievable.

**Trust assessment:** I trust the filament pricing because it comes directly from a vendor quote with specific spool counts. The budget analysis is credible but I would **verify two things**: (1) the actual PLA mass per MiniClaw unit once the CAD model is complete (~35 g estimated by FilaTech), and (2) the cost of brass pins and grip pads per unit from the component library (these are "free" internally but should still be tracked against the $3.50 target for cost accounting purposes). The Company Blue filament has a **3-week lead time** so the PO must go out by mid-April.

---
## Deliverable 5: Preliminary Design Concept / Working PRD (10 pts)

Based on your research and context work, produce a **working PRD** for the
MiniClaw. This is not a final design — it's a requirements document that will
guide MP3's detailed design phase.

**Include:**
- **8–12 product requirements** — specific, measurable, testable (as practiced
  in Session 6). For each, note the source: design brief, your research, or
  the ACME context.
- **2–3 design directions** you're considering, with brief rationale
- **Key open questions** that MP3 will need to resolve

**This document carries forward.** Your PRD will guide your detailed design
work in MP3. Write it as if your future self is the audience.

### MiniClaw Working PRD

#### Product Requirements

| # | Requirement | Source | Testable? |
|---|-------------|--------|----------|
| 1 | Grip force ≥ 5 N per jaw at 25 mm jaw opening with moderate hand effort (≤ 0.4 N·m thumb wheel torque) | Design brief + ACME-PROD-002 test method | Yes — spring-loaded force gauge at 25 mm opening (GripperBot standard protocol) |
| 2 | Jaw opening range: 0–25 mm minimum | Design brief | Yes — caliper measurement at full open |
| 3 | Overall dimensions ≤ 92 × 46 × 55 mm (80% BigClaw scale) | Design brief + ACME-VND-002 | Yes — caliper measurement of assembled unit |
| 4 | All gears: module 1.0, minimum 14 teeth, 20° pressure angle, face width ≥ 4 mm | ACME-ENG-001 gear guidelines | Yes — CAD dimension verification |
| 5 | Gear mesh contact ratio ≥ 1.5 for quiet, smooth operation | ACME-PROD-001 WidgetBot lessons (contact ratio 1.3 caused clicking) | Yes — calculated from gear geometry |
| 6 | Safety factor ≥ 2.0 for all printed gear teeth (Lewis bending stress vs. 28 MPa interlayer adhesion) | ACME-ENG-001 + ACME-MFG-003 test data | Yes — Lewis stress calculation with ACME derating |
| 7 | Design backlash: 0.10–0.15 mm for PLA gear mesh | ACME-ENG-001 | Yes — feeler gauge measurement on prototype |
| 8 | Material cost ≤ $3.50 per unit (filament + brass pins + grip pads + packaging) | ACME-BIZ-001 marketing brief | Yes — BOM cost calculation |
| 9 | Assembly time < 10 minutes per unit by trained operator, no tools required beyond flush cutters and mallet | ACME-MFG-004 assembly guidelines + design brief | Yes — timed assembly test |
| 10 | All gears printed flat on build plate at 0.15 mm layer height, 100% infill | ACME-ENG-001 print orientation + ACME-MFG-003 fatigue data | Yes — print file verification |
| 11 | Housing wall thickness ≥ 1.5 mm (2.0 mm in load-bearing areas) | ACME-VND-002 BigClaw teardown (Al walls 0.8–1.2 mm don't translate to PLA) | Yes — CAD wall thickness check |
| 12 | ACME logo embossed on exterior surface, minimum 15 mm width, readable without squinting | ACME-BIZ-001 marketing brief | Yes — visual inspection |

#### Design Directions Under Consideration

**Direction A: Single-stage spur gear, 6:1 ratio (14T pinion / 84T gear, module 1.0)**

Simplest approach — one gear mesh, fewest parts, lowest cost. A 14T/84T pair at module 1.0 gives pitch diameters of 14 mm and 84 mm, with center distance of 49 mm. The large gear diameter (86.0 mm outer) fits within the 92 mm envelope but leaves minimal clearance. Advantages: fewer parts means faster assembly and fewer tolerance contributors. Risk: the large gear may make the housing bulky, and contact ratio needs to be verified (14T at 20° pressure angle gives contact ratio ~1.6, which meets the > 1.5 requirement). Lewis bending stress at 0.3 N·m input: ~8 MPa on pinion with 4 mm face width — well within the 14 MPa allowable.

**Direction B: Two-stage spur gear reduction (e.g., 14T/42T × 16T/48T = 9:1, module 1.0)**

Follows the WidgetBot 2.0 architecture (ACME-PROD-001). Two stages allow a higher gear ratio in a more compact envelope — the largest gear is only 48T (48 mm pitch diameter), leaving more room for the linkage and housing. The WidgetBot 2.0 used this exact configuration and produced 300 units successfully, so ACME has proven manufacturing experience. Risk: more parts (4 gears vs. 2), additional tolerance stack-up at the intermediate shaft, and the WidgetBot's gear noise issue (contact ratio 1.3 on the first stage) must be addressed by increasing the pinion tooth count or using profile shift.

**Direction C: Single-stage with higher tooth count (18T/108T = 6:1, module 0.8)**

Module 0.8 is the smallest preferred module in ACME-ENG-001. Higher tooth counts (18T minimum at module 0.8) give excellent contact ratio (~1.7) and very smooth feel. Pitch diameters: 14.4 mm and 86.4 mm. However, module 0.8 teeth are finer and may be at the edge of reliable FDM printing per ACME's guidelines. Would need prototype validation. This direction prioritizes the "precision-engineered" feel that marketing wants.

#### Open Questions for MP3

- **Linkage geometry:** What crank arm length, coupler, and output link dimensions deliver near-linear force output through mid-travel? The BigClaw uses 32/28/45/38 mm (ACME-VND-002) — how do these scale to MiniClaw size?
- **Component library clarification:** Are brass pins and silicone grip pads from the component library (ACME-PROD-003) permitted under the "no purchased parts" constraint? Jordan Chen's directive implies yes, but formal confirmation is needed.
- **Print time per unit:** What is the estimated print time per MiniClaw unit on a Prusa MK4S? This determines whether the 3-week production window (May 15 – June 1) on 12 printers is sufficient for 550 units.
- **Jaw alignment tolerance:** GripperBot's top complaint was jaw misalignment > 0.5 mm (ACME-PROD-002). How do we ensure alignment across 500 units? Alignment pins on jaw arms and tight tolerance on housing pivot bores are recommended — need to verify this is achievable within ACME's ±0.15 mm press-fit tolerance.
- **Gear ratio selection:** Direction A (6:1) vs. Direction B (9:1) — a 6:1 ratio needs ~0.3 N·m input for 5 N grip force, while 9:1 needs only ~0.2 N·m. Is the higher ratio worth the added complexity? Prototype testing with both ratios would answer this.

---
## Submission

Before submitting, verify:

- [ ] All cells run top-to-bottom without errors
- [ ] All five deliverables are complete (no placeholder text remaining)
- [ ] Before/after demo shows three meaningful engineering questions
- [ ] PRD has 8–12 specific, measurable requirements
- [ ] Your assessments and evaluations reflect your own engineering judgment

**To submit:**
1. Save this notebook (Ctrl+S)
2. Open the **Source Control** panel in VS Code (left sidebar, branch icon)
3. Stage your changes, write a commit message, and **Commit & Push**
4. Verify on GitHub that your notebook appears with all outputs visible
5. Submit your repository URL on **Canvas** (same URL as Part A)

> ⚠️ **Codespace warning:** Your Codespace may be deleted after a period of
> inactivity. Always push your work to GitHub — don't rely on the Codespace
> persisting.